<a href="https://colab.research.google.com/github/abosameh/CloudHunter/blob/master/Cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Verify GPU acceleration is active
!nvidia-smi
!sudo apt-get install -y pciutils zstd
# 2. Install the official Linux Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Download the official Cloudflare Tunnel Linux client
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n Setup finished! Head to Cell 2.")

Thu Jul  2 22:55:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [4]:
import subprocess
import time
import re
import os

# Set environment variable so Ollama binds correctly to all external traffic loops
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

print("🔄 Launching Ollama Server...")
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)  # Allow system initialization

print("🌐 Spawning Cloudflare Quick Tunnel...")
# Start cloudflared to forward traffic and correctly inject host headers
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434', '--http-host-header', 'localhost:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
tunnel_url = None

# Scan cloudflared's terminal logs to find your generated public domain URL
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print(f"\n🚀 SUCCESS! Your local PC can connect to this endpoint:")
    print(f"👉 {tunnel_url} 👈")
    print("\nKeep this notebook running. Copy the URL above and pass it to your local scripts!")
else:
    print("\n❌ Failed to capture a tunnel URL automatically. Run tunnel manually or check logs.")

🔄 Launching Ollama Server...
🌐 Spawning Cloudflare Quick Tunnel...

🚀 SUCCESS! Your local PC can connect to this endpoint:
👉 https://switch-costs-showers-sponsorship.trycloudflare.com 👈

Keep this notebook running. Copy the URL above and pass it to your local scripts!


In [8]:
# Pull your models here. You can swap 'llama3.2' for 'deepseek-r1:8b' or 'gemma2'
!ollama pull devstral

In [9]:
!ollama list

NAME               ID              SIZE      MODIFIED       
devstral:latest    9bd74193e939    14 GB     7 seconds ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    19 minutes ago    


In [7]:
# Make a request to the Ollama API through the tunnel
import requests
import json

# Use your tunnel URL here
tunnel_url = "https://switch-costs-showers-sponsorship.trycloudflare.com"

response = requests.post(f"{tunnel_url}/api/generate",
                        json={
                            "model": "llama3.2:latest",
                            "prompt": "Hello, how are you?",
                            "stream": False
                        })

print(response.json()["response"])

I'm just a language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?


In [ ]:
export OLLAMA_HOST="https://switch-costs-showers-sponsorship.trycloudflare.com"